# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {getattr(dataset.metadata, 'name', '')}")
print(f"Description: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by @id
print("Record Sets in the Dataset:")
record_sets = []
if hasattr(dataset.metadata, 'record_set'):
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in dataset.metadata.record_set]
    for rs in dataset.metadata.record_set:
        if isinstance(rs, dict):
            print(f"@id: {rs.get('@id', '')} | name: {rs.get('name', '')}")
        else:
            print(f"@id: {rs}")
else:
    print("No record set found in top-level metadata. Attempting to introspect via dataset API.")
    # Try to extract from .record_sets property (mlcroissant >= 0.3.0+)
    try:
        record_sets = [r['@id'] for r in dataset.record_sets]
        for r in dataset.record_sets:
            print(f"@id: {r.get('@id', '')} | name: {r.get('name', '')}")
    except Exception as e:
        print("Could not extract record sets.")

# Attempt to get the first available record set for demonstration
# If that fails, user will need to consult the schema out-of-band
if not record_sets:
    print("Warning: No record sets available, skipping example record read.")
else:
    example_record_set = record_sets[0]
    print(f"\nFields/Columns in Record Set {example_record_set}:")
    try:
        # Print first record for this record set
        recs = dataset.records(record_set=example_record_set)
        first_record = next(recs)
        print(json.dumps(first_record, indent=2))
    except Exception as e:
        print("Unable to iterate records for the record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Here we attempt to load all available record sets (by @id)
if not record_sets:
    print("No record sets to load.")
else:
    dataframes = {}
    for rs_id in record_sets:
        print(f"Loading records for: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows for {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())  # Only works inside Jupyter notebook
        except Exception as e:
            print(f"Failed to load record set {rs_id}: {str(e)}")

    # For the sake of illustration, let's select the first loaded record set
    selected_record_set = record_sets[0]
    selected_df = dataframes[selected_record_set]
    print(f"\nUsing columns from record set {selected_record_set}: {selected_df.columns.tolist()}")
    selected_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose a numeric field present in the dataframe for analysis
# If your schema provides field @ids such as 'coverage' or 'Abundance', set those explicitly.
numeric_field_candidates = [col for col in selected_df.columns if selected_df[col].dtype in [np.float64, np.int64, float, int]]

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Using numeric field '{numeric_field}' for analysis.")
    threshold = selected_df[numeric_field].quantile(0.75)  # use 75th percentile as threshold
    filtered_df = selected_df[selected_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())  # Only works inside Jupyter notebook

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field (e.g. 'description', or another field in the dataset)
    group_field_candidates = [col for col in selected_df.columns if selected_df[col].dtype == object and col != numeric_field]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric fields detected in the selected DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_candidates:
    # Plot histogram of numeric field
    plt.figure(figsize=(8,4))
    selected_df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field available, boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        selected_df.boxplot(column=numeric_field, by=group_field, rot=45, grid=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No visualization possible, no numeric field detected.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The `mlcroissant` library makes it easy to load and explore FAIR datasets using only the schema URL and entity `@id`s. 
* We inspected the dataset's record sets, extracted sample records, performed basic EDA (such as normalization and grouping), and visualized major numeric fields.
* For advanced analysis, consult the Croissant documentation and FAIR2 schema to leverage all semantic relationships via their `@id`s.